# Build a Stance-Detection Dataset (Single Source)

**Goal:** take one raw labeled dataset and turn it into the exact format
`finetune_stance.py` needs, then push it to the HF Hub. No GPU needed.

**This notebook is self-contained on purpose.** The prompt template, label map, and
conversation-building logic are defined directly below -- not imported from
`prepare_stance_data.py` -- so you can actually read and tweak every piece of the
format while learning it. `prepare_stance_data.py` at the repo root still has its own
copy of this same logic and is what the real training runs use; if you change the
format here for good, port the change back there too.

## The format, up front

`finetune_stance.py` expects a HF `Dataset` with a **`conversations`** column.
Each row is a list of turns:

```python
[
    {"role": "user", "content": "...the prompt..."},
    {"role": "assistant", "content": '{"stance": "FAVOR"}'},
]
```

This is **ChatML-style** (`role`/`content`), *not* Unsloth's default ShareGPT style
(`from`/`value`). Why: training runs with `--no_from_foundation_model`, which applies
the tokenizer's own native chat template -- and that template expects `role`/`content`.

The `user` message text must match **exactly** the prompt used at test time in the
Tilburg vLLM eval pipeline (`stance_classification_task_definition` in
`genai_functions.py`). Any drift here is a silent train/test mismatch.


## 1. Define the prompt template + label map

This is normally the part hidden behind an import -- here it is directly:

- `STANCE_PROMPT_TEMPLATE` -- must match the Tilburg eval prompt exactly (down to
  whitespace), since the fine-tuned model is tested with this same wording.
- `STANCE_LABEL_MAP` -- different raw datasets spell labels differently (`PRO` vs
  `FAVOR`, `NEUTRAL` vs `NONE`, ...). This canonicalizes everything onto one fixed
  3-way vocabulary: `FAVOR` / `AGAINST` / `NONE`.


In [4]:
STANCE_PROMPT_TEMPLATE = (
    "Stance classification is the task of determining the expressed or implied opinion, "
    "or stance, of a document toward a certain, specified target. "
    "Analyze the following document and determine its stance toward the provided query.\n\n"
    "QUERY: {target}\n\n"
    "DOCUMENT: {text}\n\n"
    'Return valid JSON in exactly this format: {{"stance": "FAVOR"}}\n'
    'The "stance" value must be exactly one of: "FAVOR", "AGAINST", "NONE".\n'
    'Use "FAVOR" only when the author is definitely in favor of the query. '
    'Use "AGAINST" only when the author is definitely against the query. '
    'Use "NONE" if any of the following holds: (a) the document does not discuss the query '
    "at all, (b) the document discusses it but the author takes no clear side "
    "(neutral/balanced), or (c) the author's position cannot be determined with confidence. "
    "Do not guess from indirect hints.\n"
)

# Maps every label spelling seen across stance datasets onto our fixed 3-way vocabulary.
# Add an entry here whenever a new dataset uses a label spelling we haven't seen yet --
# canonicalize_label() below deliberately raises instead of silently guessing.
STANCE_LABEL_MAP = {
    "FAVOR": "FAVOR",
    "AGAINST": "AGAINST",
    "NONE": "NONE",
    "PRO": "FAVOR",
    "NEUTRAL": "NONE",
    "UNCLEAR": "NONE",
    "UNRELATED": "NONE",
    "SUPPORTS": "FAVOR",
    "DENIES": "AGAINST",
}


def canonicalize_label(raw_label):
    normalized = str(raw_label).strip().upper()
    if normalized not in STANCE_LABEL_MAP:
        raise ValueError(
            f"Unrecognized stance label {raw_label!r} -- add it to STANCE_LABEL_MAP "
            f"(known: {sorted(STANCE_LABEL_MAP)})"
        )
    return STANCE_LABEL_MAP[normalized]


def build_conversations(df, text_column, target_column, label_column):
    conversations = []
    for _, row in df.iterrows():
        prompt = STANCE_PROMPT_TEMPLATE.format(target=row[target_column], text=row[text_column])
        answer = json.dumps({"stance": canonicalize_label(row[label_column])})
        conversations.append(
            [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": answer},
            ]
        )
    return conversations


## 2. Load the raw dataset

Raw SemEval-2016 Task 6 file: tab-separated, **latin1** encoded (it has curly
quotes/apostrophes that break a plain UTF-8 read), columns `ID`, `Target`, `Tweet`, `Stance`.


In [8]:
from pathlib import Path
import json
import pandas as pd

RAW_PATH = Path(
    "/Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper"
    "/data_in/semeval/semeval2016-task6-trainingdata.txt"
)

df = pd.read_csv(RAW_PATH, sep="\t", encoding="latin1")
print(df.shape)
df.head()


(2814, 4)


,ID,Target,Tweet,Stance
0,101,Atheism,dear lord thank u for all of ur blessings forg...,AGAINST
1,102,Atheism,"Blessed are the peacemakers, for they shall be...",AGAINST
2,103,Atheism,I am not conformed to this world. I am transfo...,AGAINST
3,104,Atheism,Salah should be prayed with #focus and #unders...,AGAINST
4,105,Atheism,And stay in your houses and do not display you...,AGAINST


## 3. Check the raw label distribution before canonicalizing anything


In [6]:
df["Stance"].value_counts()


Stance
AGAINST    1342
NONE        741
FAVOR       731
Name: count, dtype: int64

## 4. Build the `conversations` column


In [9]:
conversations = build_conversations(
    df,
    text_column="Tweet",
    target_column="Target",
    label_column="Stance",
)
print(f"Built {len(conversations)} conversation examples")
conversations[0]  # this is exactly what one training row looks like


Built 2814 conversation examples


[{'role': 'user',
  'content': 'Stance classification is the task of determining the expressed or implied opinion, or stance, of a document toward a certain, specified target. Analyze the following document and determine its stance toward the provided query.\n\nQUERY: Atheism\n\nDOCUMENT: dear lord thank u for all of ur blessings forgive my sins lord give me strength and energy for this busy day ahead #blessed #hope #SemST\n\nReturn valid JSON in exactly this format: {"stance": "FAVOR"}\nThe "stance" value must be exactly one of: "FAVOR", "AGAINST", "NONE".\nUse "FAVOR" only when the author is definitely in favor of the query. Use "AGAINST" only when the author is definitely against the query. Use "NONE" if any of the following holds: (a) the document does not discuss the query at all, (b) the document discusses it but the author takes no clear side (neutral/balanced), or (c) the author\'s position cannot be determined with confidence. Do not guess from indirect hints.\n'},
 {'role': '

## 5. (Recommended) Preview the exact text the model will train on

This applies the real Qwen3 chat template locally -- no GPU needed, it just downloads
the tokenizer config. Best way to *see* the format instead of imagining it.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3-1.7B-unsloth-bnb-4bit")
rendered = tokenizer.apply_chat_template(conversations[0], tokenize=False)
print(rendered)


## 6. Convert to a Hugging Face `Dataset` and sanity-check


In [10]:
from datasets import Dataset

dataset = Dataset.from_dict({"conversations": conversations})

assert "conversations" in dataset.column_names
assert all(
    turn["role"] in ("user", "assistant")
    for convo in dataset["conversations"][:5]
    for turn in convo
)
print("Format OK")
dataset


/Users/nityaakalra/Desktop/fine_tune/fine-tuning-slms/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/nityaakalra/Desktop/fine_tune/fine-tuning-slms/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Format OK


Dataset({
    features: ['conversations'],
    num_rows: 2814
})

## 7. Push to the Hugging Face Hub

Requires `hf auth login` to already be done in this environment. Keep `private=True`
while iterating; flip to `False` only for the final dataset you reference publicly.


In [11]:
PUSH_TO_HUB_ID = "nityaak/semeval-stance-conversations"  # change if this is a new/different dataset
PRIVATE = True

# Uncomment when ready:
dataset.push_to_hub(PUSH_TO_HUB_ID, private=PRIVATE)
print(f"Pushed to https://huggingface.co/datasets/{PUSH_TO_HUB_ID}")


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 171.41ba/s]
Processing Files (1 / 1): 100%|██████████|  389kB /  389kB, 37.2kB/s  
New Data Upload: 100%|██████████|  389kB /  389kB, 37.2kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.66s/ shards]


Pushed to https://huggingface.co/datasets/nityaak/semeval-stance-conversations


## Recap: what "correct format" means for this project

- Column name: `conversations`
- Each row: list of `{role, content}` dicts, `role` in `{user, assistant}`
- `user` content = the exact eval prompt wording (QUERY / DOCUMENT / JSON instructions)
- `assistant` content = `{"stance": "FAVOR"|"AGAINST"|"NONE"}` as a JSON **string**
- No ShareGPT `from`/`value` -- that's Unsloth's *other* default template, not used here
  because of `--no_from_foundation_model`
- This notebook keeps its own copy of the prompt/label logic (not imported) so you can
  see it directly -- `prepare_stance_data.py` is the copy the real pipeline runs off of
